# Wallet Signal Strategy v2

This notebook rebuilds the wallet signal pipeline around **edge**, not hit rate.

Main changes:
- wallet skill is measured on `open_buy` events only
- metric selection uses `train_a -> train_b`, keeping the final test set untouched
- signals become a continuous score instead of a few hard-coded buckets
- final evaluation uses realistic forward fills on the held-out test set


In [208]:
import datetime
from collections import defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [209]:
END_DATE_TRAIN = datetime.date(2026, 2, 10)
TRAIN_START_DATE = datetime.date(2025, 1, 1)
TRAIN_A_END_DATE = TRAIN_START_DATE + (END_DATE_TRAIN - TRAIN_START_DATE) // 2
RECENCY_DAYS = 90
PRICE_BUCKET_BINS = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    '0.00-0.10', '0.10-0.25', '0.25-0.40', '0.40-0.60',
    '0.60-0.75', '0.75-0.90', '0.90-1.00',
]

KELLY_SCALE = 0.25
KELLY_MAX_FRACTION = 0.10
KELLY_MIN_STAKE_USDC = 25.0
KELLY_MAX_STAKE_USDC = 750.0
COHORT_TOP_N_DEFAULT = 100
FILL_MAX_REL_PRICE_DIFF_BY_BUCKET = {
    '0.00-0.10': 2.50,
    '0.10-0.25': 1.20,
    '0.25-0.40': 0.70,
    '0.40-0.60': 0.35,
    '0.60-0.75': 0.22,
    '0.75-0.90': 0.12,
    '0.90-1.00': 0.06,
}

TRADES_DIR = Path('../data/polygon_trades_processed')#/0.parquet')
WORKSPACE_DIR = Path('../data/trade_signals_workspace_v2')
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format='parquet')
DATASET_COLUMNS = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = 'transaction_hash' if 'transaction_hash' in DATASET_COLUMNS else ('tx_hash' if 'tx_hash' in DATASET_COLUMNS else None)
FILL_TX_HASH_COLUMN = TRIGGER_TX_HASH_COLUMN

METRICS_A_PATH = WORKSPACE_DIR / 'wallet_metrics_train_a.parquet'
METRICS_B_PATH = WORKSPACE_DIR / 'wallet_metrics_train_b.parquet'
METRICS_FULL_PATH = WORKSPACE_DIR / 'wallet_metrics_full_train.parquet'
METRICS_TEST_PATH = WORKSPACE_DIR / 'wallet_metrics_test.parquet'
SELECTED_WALLETS_PATH = WORKSPACE_DIR / 'selected_wallets_v2.parquet'
SELECTED_PROFILES_PATH = WORKSPACE_DIR / 'selected_wallet_profiles_v2.parquet'
CALIBRATION_SIGNALS_PATH = WORKSPACE_DIR / 'signal_events_train_b.parquet'
TEST_SIGNALS_PATH = WORKSPACE_DIR / 'signal_events_test.parquet'
EXEC_TAPE_TEST_PATH = WORKSPACE_DIR / 'execution_tape_test_v2.parquet'
EXEC_TAPE_TRAIN_B_PATH = WORKSPACE_DIR / 'execution_tape_train_b_v2.parquet'

print({'train_a_end': TRAIN_A_END_DATE, 'train_end': END_DATE_TRAIN, 'workspace': str(WORKSPACE_DIR)})


{'train_a_end': datetime.date(2025, 7, 22), 'train_end': datetime.date(2026, 2, 10), 'workspace': '../data/trade_signals_workspace_v2'}


## 1. Wallet skill metrics

We score wallets on `open_buy` events only, because those are the events we may want to follow live.

Candidate metrics:
- `avg_prob_edge`: mean of `final_price - entry_price`
- `weighted_prob_edge`: dollar-weighted version of the same edge
- `avg_copy_roi_capped`: realized copy-trade ROI, capped to avoid tiny-price outliers dominating
- `edge_sharpe`, `roi_sharpe`, `brier_skill`
- `hit_rate` only as a baseline


In [210]:
def aggregate_wallet_trades(df):
    if df.empty:
        return df.copy()
    required = ['wallet', 'condition_id', 'outcome', 'dt', 'side', 'quantity', 'price', 'usdc_amount']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns for aggregation: {missing}')
    work = df.copy()
    work['quantity'] = work['quantity'].astype(float)
    work['usdc_amount'] = work['usdc_amount'].astype(float)
    work['price_x_qty'] = work['price'].astype(float) * work['quantity']

    agg_map = {
        'quantity': ('quantity', 'sum'),
        'usdc_amount': ('usdc_amount', 'sum'),
        'price_x_qty': ('price_x_qty', 'sum'),
    }
    passthrough_first = [
        'final_price',
        'token_winner',
        'trigger_tx_hash',
    ]
    for col in passthrough_first:
        if col in work.columns:
            agg_map[col] = (col, 'first')
    if 'trade_pnl' in work.columns:
        agg_map['trade_pnl'] = ('trade_pnl', 'sum')

    grouped = (
        work.groupby(['wallet', 'condition_id', 'outcome', 'dt', 'side'], as_index=False)
        .agg(**agg_map)
        .sort_values(['wallet', 'condition_id', 'outcome', 'dt', 'side'])
        .reset_index(drop=True)
    )
    grouped['price'] = grouped['price_x_qty'] / grouped['quantity'].clip(lower=1e-9)
    grouped = grouped.drop(columns=['price_x_qty'])

    grouped['signed_quantity'] = np.where(grouped['side'] == 'BUY', grouped['quantity'], -grouped['quantity'])
    grouped['side_order'] = np.where(grouped['side'] == 'BUY', 0, 1)
    grouped = grouped.sort_values(['wallet', 'condition_id', 'outcome', 'dt', 'side_order']).reset_index(drop=True)
    grouped['position'] = grouped.groupby(['wallet', 'condition_id', 'outcome'])['signed_quantity'].cumsum()
    grouped['prev_position'] = grouped['position'] - grouped['signed_quantity']
    return grouped.drop(columns=['side_order'])


def _new_wallet_stat():
    return {
        'open_buy_trades': 0,
        'volume': 0.0,
        'wins': 0,
        'sum_prob_edge': 0.0,
        'sum_prob_edge_sq': 0.0,
        'sum_weighted_edge_num': 0.0,
        'sum_copy_roi': 0.0,
        'sum_copy_roi_sq': 0.0,
        'sum_copy_roi_capped': 0.0,
        'sum_copy_roi_capped_sq': 0.0,
        'sum_copy_pnl_usdc': 0.0,
        'sum_trade_pnl': 0.0,
        'sum_brier': 0.0,
        'sum_price': 0.0,
    }


def _finalize_metric_frame(stats, period_name, baseline_brier, market_counts=None, recent_stats=None, edge_prior_trades=25.0, edge_prior_volume=2500.0):
    rows = []
    for wallet, wallet_stats in stats.items():
        s = wallet_stats[period_name]
        n = s['open_buy_trades']
        volume = s['volume']
        avg_prob_edge = s['sum_prob_edge'] / n if n else np.nan
        weighted_prob_edge = s['sum_weighted_edge_num'] / volume if volume else np.nan
        avg_copy_roi = s['sum_copy_roi'] / n if n else np.nan
        avg_copy_roi_capped = s['sum_copy_roi_capped'] / n if n else np.nan
        hit_rate = s['wins'] / n if n else np.nan
        mean_brier = s['sum_brier'] / n if n else np.nan
        prob_edge_var = (s['sum_prob_edge_sq'] / n - avg_prob_edge ** 2) if n else np.nan
        copy_roi_var = (s['sum_copy_roi_capped_sq'] / n - avg_copy_roi_capped ** 2) if n else np.nan
        prob_edge_std = np.sqrt(max(prob_edge_var, 0.0)) if n else np.nan
        copy_roi_std = np.sqrt(max(copy_roi_var, 0.0)) if n else np.nan
        rows.append({
            'wallet': wallet,
            'period': period_name,
            'open_buy_trades': n,
            'volume': volume,
            'wins': s['wins'],
            'hit_rate': hit_rate,
            'avg_price': s['sum_price'] / n if n else np.nan,
            'avg_prob_edge': avg_prob_edge,
            'weighted_prob_edge': weighted_prob_edge,
            'prob_edge_shrunk': s['sum_prob_edge'] / (n + edge_prior_trades) if n or edge_prior_trades else 0.0,
            'weighted_prob_edge_shrunk': s['sum_weighted_edge_num'] / (volume + edge_prior_volume) if volume or edge_prior_volume else 0.0,
            'avg_copy_roi': avg_copy_roi,
            'avg_copy_roi_capped': avg_copy_roi_capped,
            'copy_roi_std': copy_roi_std,
            'edge_sharpe': avg_prob_edge / prob_edge_std if n and prob_edge_std > 1e-12 else np.nan,
            'roi_sharpe': avg_copy_roi_capped / copy_roi_std if n and copy_roi_std > 1e-12 else np.nan,
            'mean_brier': mean_brier,
            'brier_skill': 1.0 - (mean_brier / baseline_brier) if n and baseline_brier > 1e-12 else np.nan,
            'sum_prob_edge': s['sum_prob_edge'],
            'sum_weighted_edge_num': s['sum_weighted_edge_num'],
            'sum_copy_roi_capped': s['sum_copy_roi_capped'],
            'total_copy_pnl_usdc': s['sum_copy_pnl_usdc'],
            'total_trade_pnl_usdc': s['sum_trade_pnl'],
            'pnl_per_open_buy': s['sum_copy_pnl_usdc'] / n if n else np.nan,
            'pnl_per_1k_volume': s['sum_copy_pnl_usdc'] * 1000.0 / volume if volume else np.nan,
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    if market_counts is not None:
        df['distinct_markets'] = df['wallet'].map(market_counts).fillna(0).astype(int)
    if recent_stats is not None:
        df['recent_open_buy_trades'] = df['wallet'].map(lambda w: recent_stats.get(w, {}).get('open_buy_trades', 0)).fillna(0).astype(int)
        df['recent_volume'] = df['wallet'].map(lambda w: recent_stats.get(w, {}).get('volume', 0.0)).fillna(0.0)
    return df.sort_values(['open_buy_trades', 'volume'], ascending=[False, False]).reset_index(drop=True)


def compute_wallet_skill_workspace(dataset, batch_size=300_000):
    periods = ('train_a', 'train_b', 'full_train', 'test')
    stats = defaultdict(lambda: {period: _new_wallet_stat() for period in periods})
    market_sets_full_train = defaultdict(set)
    recent_stats = defaultdict(lambda: {'open_buy_trades': 0, 'volume': 0.0})
    baseline = {period: {'sum_brier': 0.0, 'n': 0} for period in periods}
    recent_cutoff = pd.Timestamp(END_DATE_TRAIN, tz='UTC') - pd.Timedelta(days=RECENCY_DAYS)
    columns = ['wallet', 'condition_id', 'outcome', 'dt', 'side', 'quantity', 'price', 'usdc_amount', 'trade_pnl', 'final_price', 'token_winner']

    for batch in dataset.to_batches(columns=columns, batch_size=batch_size):
        df = batch.to_pandas()
        if df.empty:
            continue
        df['dt'] = pd.to_datetime(df['dt'], utc=True)
        df = aggregate_wallet_trades(df)
        df = df[(df['side'] == 'BUY') & (df['prev_position'] <= 1e-9)].copy()
        if df.empty:
            continue
        trade_date = df['dt'].dt.date
        df['period'] = np.where(
            trade_date <= TRAIN_A_END_DATE,
            'train_a',
            np.where(trade_date <= END_DATE_TRAIN, 'train_b', 'test'),
        )
        df['prob_edge'] = df['final_price'] - df['price']
        df['weighted_edge_num'] = df['prob_edge'] * df['usdc_amount']
        raw_copy_roi = np.where(df['token_winner'], 1.0 / df['price'].clip(lower=0.001) - 1.0, -1.0)
        df['copy_roi'] = raw_copy_roi
        df['copy_roi_capped'] = np.clip(raw_copy_roi, -1.0, 10.0)
        df['copy_pnl_usdc'] = df['copy_roi'] * df['usdc_amount']
        if 'trade_pnl' not in df.columns:
            df['trade_pnl'] = np.nan
        df['brier'] = (df['price'] - df['final_price']) ** 2
        grouped = (
            df.groupby(['wallet', 'period'])
            .agg(
                open_buy_trades=('wallet', 'size'),
                volume=('usdc_amount', 'sum'),
                wins=('token_winner', 'sum'),
                sum_prob_edge=('prob_edge', 'sum'),
                sum_prob_edge_sq=('prob_edge', lambda s: float(np.square(s).sum())),
                sum_weighted_edge_num=('weighted_edge_num', 'sum'),
                sum_copy_roi=('copy_roi', 'sum'),
                sum_copy_roi_sq=('copy_roi', lambda s: float(np.square(s).sum())),
                sum_copy_roi_capped=('copy_roi_capped', 'sum'),
                sum_copy_roi_capped_sq=('copy_roi_capped', lambda s: float(np.square(s).sum())),
                sum_copy_pnl_usdc=('copy_pnl_usdc', 'sum'),
                sum_trade_pnl=('trade_pnl', 'sum'),
                sum_brier=('brier', 'sum'),
                sum_price=('price', 'sum'),
            )
            .reset_index()
        )
        for row in grouped.itertuples(index=False):
            d = stats[row.wallet][row.period]
            d['open_buy_trades'] += int(row.open_buy_trades)
            d['volume'] += float(row.volume)
            d['wins'] += int(row.wins)
            d['sum_prob_edge'] += float(row.sum_prob_edge)
            d['sum_prob_edge_sq'] += float(row.sum_prob_edge_sq)
            d['sum_weighted_edge_num'] += float(row.sum_weighted_edge_num)
            d['sum_copy_roi'] += float(row.sum_copy_roi)
            d['sum_copy_roi_sq'] += float(row.sum_copy_roi_sq)
            d['sum_copy_roi_capped'] += float(row.sum_copy_roi_capped)
            d['sum_copy_roi_capped_sq'] += float(row.sum_copy_roi_capped_sq)
            d['sum_copy_pnl_usdc'] += float(row.sum_copy_pnl_usdc)
            d['sum_trade_pnl'] += float(row.sum_trade_pnl)
            d['sum_brier'] += float(row.sum_brier)
            d['sum_price'] += float(row.sum_price)
            if row.period in ('train_a', 'train_b'):
                full = stats[row.wallet]['full_train']
                full['open_buy_trades'] += int(row.open_buy_trades)
                full['volume'] += float(row.volume)
                full['wins'] += int(row.wins)
                full['sum_prob_edge'] += float(row.sum_prob_edge)
                full['sum_prob_edge_sq'] += float(row.sum_prob_edge_sq)
                full['sum_weighted_edge_num'] += float(row.sum_weighted_edge_num)
                full['sum_copy_roi'] += float(row.sum_copy_roi)
                full['sum_copy_roi_sq'] += float(row.sum_copy_roi_sq)
                full['sum_copy_roi_capped'] += float(row.sum_copy_roi_capped)
                full['sum_copy_roi_capped_sq'] += float(row.sum_copy_roi_capped_sq)
                full['sum_copy_pnl_usdc'] += float(row.sum_copy_pnl_usdc)
                full['sum_trade_pnl'] += float(row.sum_trade_pnl)
                full['sum_brier'] += float(row.sum_brier)
                full['sum_price'] += float(row.sum_price)
        baseline_grouped = df.groupby('period').agg(sum_brier=('brier', 'sum'), n=('wallet', 'size')).reset_index()
        for row in baseline_grouped.itertuples(index=False):
            baseline[row.period]['sum_brier'] += float(row.sum_brier)
            baseline[row.period]['n'] += int(row.n)
            if row.period in ('train_a', 'train_b'):
                baseline['full_train']['sum_brier'] += float(row.sum_brier)
                baseline['full_train']['n'] += int(row.n)
        full_train_df = df[df['period'].isin(['train_a', 'train_b'])]
        if not full_train_df.empty:
            pairs = full_train_df[['wallet', 'condition_id']].drop_duplicates()
            for row in pairs.itertuples(index=False):
                market_sets_full_train[row.wallet].add(row.condition_id)
            recent_df = full_train_df[full_train_df['dt'] >= recent_cutoff]
            if not recent_df.empty:
                recent_grouped = recent_df.groupby('wallet').agg(open_buy_trades=('wallet', 'size'), volume=('usdc_amount', 'sum')).reset_index()
                for row in recent_grouped.itertuples(index=False):
                    recent_stats[row.wallet]['open_buy_trades'] += int(row.open_buy_trades)
                    recent_stats[row.wallet]['volume'] += float(row.volume)

    baseline_brier = {period: (values['sum_brier'] / values['n'] if values['n'] else np.nan) for period, values in baseline.items()}
    market_counts = {wallet: len(markets) for wallet, markets in market_sets_full_train.items()}
    train_a = _finalize_metric_frame(stats, 'train_a', baseline_brier['train_a'])
    train_b = _finalize_metric_frame(stats, 'train_b', baseline_brier['train_b'])
    full_train = _finalize_metric_frame(stats, 'full_train', baseline_brier['full_train'], market_counts=market_counts, recent_stats=recent_stats)
    test = _finalize_metric_frame(stats, 'test', baseline_brier['test'])
    return train_a, train_b, full_train, test


In [211]:
if all(path.exists() for path in [METRICS_A_PATH, METRICS_B_PATH, METRICS_FULL_PATH, METRICS_TEST_PATH]):
    train_a_metrics = pd.read_parquet(METRICS_A_PATH)
    train_b_metrics = pd.read_parquet(METRICS_B_PATH)
    full_train_metrics = pd.read_parquet(METRICS_FULL_PATH)
    test_metrics = pd.read_parquet(METRICS_TEST_PATH)
else:
    train_a_metrics, train_b_metrics, full_train_metrics, test_metrics = compute_wallet_skill_workspace(dataset)
    train_a_metrics.to_parquet(METRICS_A_PATH, index=False)
    train_b_metrics.to_parquet(METRICS_B_PATH, index=False)
    full_train_metrics.to_parquet(METRICS_FULL_PATH, index=False)
    test_metrics.to_parquet(METRICS_TEST_PATH, index=False)

pd.Series({
    'train_a_wallets': len(train_a_metrics),
    'train_b_wallets': len(train_b_metrics),
    'full_train_wallets': len(full_train_metrics),
    'test_wallets': len(test_metrics),
}).to_frame('value')


,value
train_a_wallets,45407
train_b_wallets,45407
full_train_wallets,45407
test_wallets,45407


## 2. Persistence and metric selection

In [212]:
CANDIDATE_METRICS = [
    'prob_edge_shrunk',
    'weighted_prob_edge_shrunk',
    'avg_copy_roi_capped',
    'edge_sharpe',
    'roi_sharpe',
    'brier_skill',
    'hit_rate',
]

paired_metrics = train_a_metrics.merge(train_b_metrics, on='wallet', suffixes=('_a', '_b'), how='inner')
paired_metrics = paired_metrics[(paired_metrics['open_buy_trades_a'] >= 10) & (paired_metrics['open_buy_trades_b'] >= 10)].copy()

persistence_rows = []
for metric in CANDIDATE_METRICS:
    a_col = f'{metric}_a'
    b_col = f'{metric}_b'
    persistence_rows.append({
        'metric': metric,
        'wallets': len(paired_metrics[[a_col, b_col]].dropna()),
        'pearson': paired_metrics[a_col].corr(paired_metrics[b_col]),
        'spearman': paired_metrics[a_col].corr(paired_metrics[b_col], method='spearman'),
    })

persistence_summary = pd.DataFrame(persistence_rows).sort_values(['spearman', 'pearson'], ascending=False)
persistence_summary


,metric,wallets,pearson,spearman
5,brier_skill,2885,0.649407,0.660294
6,hit_rate,2885,0.653742,0.613655
2,avg_copy_roi_capped,2885,0.101128,0.110286
4,roi_sharpe,2881,-0.004399,0.104935
3,edge_sharpe,2885,0.025560,0.060963
0,prob_edge_shrunk,2885,0.061236,0.049178
1,weighted_prob_edge_shrunk,2885,-0.075014,-0.073217


In [213]:
def cohort_selection_sweep(train_a_df, train_b_df, metrics, top_ns=(50, 100, 200, 300, 500)):
    eligible_a = train_a_df[(train_a_df['open_buy_trades'] >= 10) & (train_a_df['volume'] >= 250)].copy()
    results = []
    for metric in metrics:
        ranked = eligible_a.sort_values(metric, ascending=False).dropna(subset=[metric])
        for top_n in top_ns:
            selected = set(ranked.head(top_n)['wallet'])
            cohort = train_b_df[train_b_df['wallet'].isin(selected)].copy()
            if cohort.empty:
                continue
            trades = cohort['open_buy_trades'].sum()
            volume = cohort['volume'].sum()
            results.append({
                'metric': metric,
                'top_n': top_n,
                'wallets_found_in_train_b': cohort['wallet'].nunique(),
                'train_b_open_buy_trades': trades,
                'train_b_weighted_prob_edge': cohort['sum_weighted_edge_num'].sum() / volume if volume else np.nan,
                'train_b_avg_prob_edge': cohort['sum_prob_edge'].sum() / trades if trades else np.nan,
                'train_b_avg_copy_roi_capped': cohort['sum_copy_roi_capped'].sum() / trades if trades else np.nan,
                'train_b_total_copy_pnl_usdc': cohort['total_copy_pnl_usdc'].sum(),
                'train_b_hit_rate': cohort['wins'].sum() / trades if trades else np.nan,
            })
    return pd.DataFrame(results)

cohort_sweep = cohort_selection_sweep(train_a_metrics, train_b_metrics, CANDIDATE_METRICS)
cohort_sweep.sort_values(['train_b_avg_copy_roi_capped', 'train_b_weighted_prob_edge'], ascending=False).head(20)


,metric,top_n,wallets_found_in_train_b,train_b_open_buy_trades,train_b_weighted_prob_edge,train_b_avg_prob_edge,train_b_avg_copy_roi_capped,train_b_total_copy_pnl_usdc,train_b_hit_rate
0,prob_edge_shrunk,50,50,2188,0.084714,0.144695,0.901704,100566.740631,0.417733
1,prob_edge_shrunk,100,100,3418,-0.005363,0.103747,0.597762,32786.499393,0.447630
2,prob_edge_shrunk,200,200,10178,0.000995,0.084642,0.438224,72766.808891,0.470132
13,avg_copy_roi_capped,300,300,9398,0.030304,0.066977,0.410752,174619.921275,0.433922
12,avg_copy_roi_capped,200,200,4473,0.036930,0.059738,0.384170,128404.802915,0.418735
3,prob_edge_shrunk,300,300,13101,0.012600,0.079378,0.370006,166870.171841,0.496756
14,avg_copy_roi_capped,500,500,13994,0.027761,0.042974,0.324798,260347.157546,0.454838
10,avg_copy_roi_capped,50,50,2141,0.015695,0.051842,0.307529,19774.525120,0.410556
11,avg_copy_roi_capped,100,100,2705,0.025123,0.047399,0.266803,51251.827690,0.419593
4,prob_edge_shrunk,500,500,26621,0.029551,0.061264,0.222028,640161.293125,0.513805


In [214]:
best_row = cohort_sweep.sort_values(
    ['train_b_avg_copy_roi_capped', 'train_b_weighted_prob_edge', 'train_b_open_buy_trades'],
    ascending=[False, False, False],
).iloc[0]
BEST_SELECTION_METRIC = best_row['metric']
BEST_TOP_N = int(best_row['top_n'])
{'best_metric': BEST_SELECTION_METRIC, 'best_top_n': BEST_TOP_N}


{'best_metric': 'prob_edge_shrunk', 'best_top_n': 50}

## 3. Wallet selection on full train

In [215]:
def select_wallets(metric_df, metric_name, top_n, min_open_buys=20, min_volume=500.0, min_markets=8, min_recent_trades=3):
    distinct_markets = metric_df['distinct_markets'] if 'distinct_markets' in metric_df.columns else pd.Series(0, index=metric_df.index)
    recent_open_buy_trades = metric_df['recent_open_buy_trades'] if 'recent_open_buy_trades' in metric_df.columns else pd.Series(0, index=metric_df.index)
    selected = metric_df[
        (metric_df['open_buy_trades'] >= min_open_buys)
        & (metric_df['volume'] >= min_volume)
        & (distinct_markets >= min_markets)
        & (recent_open_buy_trades >= min_recent_trades)
    ].copy()
    selected = selected.sort_values(metric_name, ascending=False).head(top_n).reset_index(drop=True)
    if not selected.empty:
        selected['wallet_quality'] = selected[metric_name].rank(method='first', pct=True)
    return selected

selected_wallets = select_wallets(full_train_metrics, BEST_SELECTION_METRIC, BEST_TOP_N)
selected_wallets.to_parquet(SELECTED_WALLETS_PATH, index=False)
selected_wallets[['wallet', 'open_buy_trades', 'distinct_markets', 'recent_open_buy_trades', BEST_SELECTION_METRIC, 'wallet_quality']].head(15)


,wallet,open_buy_trades,distinct_markets,recent_open_buy_trades,prob_edge_shrunk,wallet_quality
0,0xbb0bd109b9f0c2a59b8819c466f064cf65ab3790,532,506,529,0.304533,1.00
1,0x5ad5c4608c4661361b91c92e1091d2c5b43c37b9,772,743,769,0.300616,0.98
2,0x27b713fc1c89d498f19977c8bc7788a161fb7710,787,23,787,0.294186,0.96
3,0x4d7fad0c5944fc24d4a67110f8e31abd5f559485,218,210,170,0.290809,0.94
4,0x54a2c4cfc4332d831acc3f5a860d6540982c1d43,208,199,208,0.285536,0.92
5,0x82307f44c9405e73dc1cff466073dcc505535121,389,368,389,0.261187,0.90
6,0xb9bca9fa4069228a37b583d64ff75efdf36b3498,21,21,21,0.248022,0.88
7,0xf1f06f49be8ce5681752ae80e660aeaace6858df,308,302,92,0.235090,0.86
8,0xc178402031235263f78c1a43bba8cd49d2be35b3,119,114,12,0.220636,0.84
9,0xfd4263b3ad08226034fe1b1ea678a46d80b58895,209,203,209,0.219044,0.82


In [216]:
def build_wallet_profiles(dataset, selected_wallets, period='full_train', batch_size=300_000):
    selected_wallet_set = set(selected_wallets['wallet'])
    size_lists = defaultdict(list)
    entry_price_lists = defaultdict(list)
    columns = ['wallet', 'condition_id', 'outcome', 'dt', 'side', 'quantity', 'usdc_amount', 'price']
    for batch in dataset.to_batches(columns=columns, batch_size=batch_size):
        df = batch.to_pandas()
        if df.empty:
            continue
        df['dt'] = pd.to_datetime(df['dt'], utc=True)
        if period == 'full_train':
            df = df[df['dt'].dt.date <= END_DATE_TRAIN]
        elif period == 'train_a':
            df = df[df['dt'].dt.date <= TRAIN_A_END_DATE]
        elif period == 'train_b':
            df = df[(df['dt'].dt.date > TRAIN_A_END_DATE) & (df['dt'].dt.date <= END_DATE_TRAIN)]
        elif period == 'test':
            df = df[df['dt'].dt.date > END_DATE_TRAIN]
        df = df[df['wallet'].isin(selected_wallet_set)].copy()
        if df.empty:
            continue
        df = aggregate_wallet_trades(df)
        df = df[(df['side'] == 'BUY') & (df['prev_position'] <= 1e-9)].copy()
        if df.empty:
            continue
        for row in df[['wallet', 'usdc_amount', 'price']].itertuples(index=False):
            size_lists[row.wallet].append(float(row.usdc_amount))
            entry_price_lists[row.wallet].append(float(row.price))
    profiles = selected_wallets[['wallet', 'wallet_quality']].copy()
    profiles['median_open_buy_usdc'] = profiles['wallet'].map(lambda w: float(np.median(size_lists[w])) if size_lists[w] else np.nan)
    profiles['mean_open_buy_usdc'] = profiles['wallet'].map(lambda w: float(np.mean(size_lists[w])) if size_lists[w] else np.nan)
    profiles['mean_open_buy_price'] = profiles['wallet'].map(lambda w: float(np.mean(entry_price_lists[w])) if entry_price_lists[w] else np.nan)
    return profiles

if SELECTED_PROFILES_PATH.exists():
    selected_wallet_profiles = pd.read_parquet(SELECTED_PROFILES_PATH)
else:
    selected_wallet_profiles = build_wallet_profiles(dataset, selected_wallets, period='full_train')
    selected_wallet_profiles.to_parquet(SELECTED_PROFILES_PATH, index=False)

print(len(selected_wallets))
selected_wallet_profiles.head()


50


,wallet,wallet_quality,median_open_buy_usdc,mean_open_buy_usdc,mean_open_buy_price
0,0xbb0bd109b9f0c2a59b8819c466f064cf65ab3790,1.00,1.500000,25.860400,0.096570
1,0x5ad5c4608c4661361b91c92e1091d2c5b43c37b9,0.98,1.469241,13.366280,0.087318
2,0x27b713fc1c89d498f19977c8bc7788a161fb7710,0.96,27.959100,21.011820,0.620230
3,0x4d7fad0c5944fc24d4a67110f8e31abd5f559485,0.94,2.000000,9.773511,0.157493
4,0x54a2c4cfc4332d831acc3f5a860d6540982c1d43,0.92,1.428113,13.033664,0.074375


## 4. Signal event construction

In [217]:
def attach_consensus_features(open_buys):
    open_buys = open_buys.sort_values(['dt', 'wallet', 'condition_id', 'outcome']).reset_index(drop=True)
    seen = defaultdict(lambda: defaultdict(set))
    recent = defaultdict(lambda: defaultdict(deque))
    same_any = []
    opp_any = []
    same_24h = []
    opp_24h = []
    for row in open_buys[['condition_id', 'outcome', 'wallet', 'dt']].itertuples(index=False):
        market_recent = recent[row.condition_id]
        cutoff = row.dt - pd.Timedelta(hours=24)
        for outcome_name in list(market_recent.keys()):
            dq = market_recent[outcome_name]
            while dq and dq[0][0] < cutoff:
                dq.popleft()
        same_any_wallets = seen[row.condition_id][row.outcome] - {row.wallet}
        opp_any_wallets = set()
        for outcome_name, wallets in seen[row.condition_id].items():
            if outcome_name != row.outcome:
                opp_any_wallets |= (wallets - {row.wallet})
        same_recent_wallets = {wallet for _, wallet in market_recent[row.outcome] if wallet != row.wallet}
        opp_recent_wallets = set()
        for outcome_name, dq in market_recent.items():
            if outcome_name != row.outcome:
                opp_recent_wallets |= {wallet for _, wallet in dq if wallet != row.wallet}
        same_any.append(len(same_any_wallets))
        opp_any.append(len(opp_any_wallets))
        same_24h.append(len(same_recent_wallets))
        opp_24h.append(len(opp_recent_wallets))
        seen[row.condition_id][row.outcome].add(row.wallet)
        market_recent[row.outcome].append((row.dt, row.wallet))
    out = open_buys.copy()
    out['prior_same_any'] = same_any
    out['prior_opp_any'] = opp_any
    out['prior_same_24h'] = same_24h
    out['prior_opp_24h'] = opp_24h
    out['consensus_velocity_24h'] = out['prior_same_24h'] / 24.0
    return out


def build_signal_events(dataset, wallet_profiles, period='test', batch_size=300_000):
    selected_wallet_set = set(wallet_profiles['wallet'])
    chunks = []
    columns = ['wallet', 'condition_id', 'dt', 'side', 'outcome', 'quantity', 'price', 'usdc_amount', 'trade_pnl', 'token_winner', 'final_price']
    tx_hash_column = TRIGGER_TX_HASH_COLUMN if 'TRIGGER_TX_HASH_COLUMN' in globals() else None
    if tx_hash_column is not None:
        columns.append(tx_hash_column)
    for batch in dataset.to_batches(columns=columns, batch_size=batch_size):
        df = batch.to_pandas()
        if df.empty:
            continue
        df['dt'] = pd.to_datetime(df['dt'], utc=True)
        if period == 'train_b':
            df = df[(df['dt'].dt.date > TRAIN_A_END_DATE) & (df['dt'].dt.date <= END_DATE_TRAIN)]
        elif period == 'test':
            df = df[df['dt'].dt.date > END_DATE_TRAIN]
        elif period == 'full_train':
            df = df[df['dt'].dt.date <= END_DATE_TRAIN]
        df = df[df['wallet'].isin(selected_wallet_set)].copy()
        if df.empty:
            continue
        if tx_hash_column is not None and tx_hash_column in df.columns:
            df['trigger_tx_hash'] = df[tx_hash_column].astype(str)
        df = aggregate_wallet_trades(df)
        df['position_change'] = df['signed_quantity']
        df['market_key'] = df['condition_id'] + '|' + df['outcome']
        df['event_type'] = np.select(
            [
                (df['side'] == 'BUY') & (df['prev_position'] <= 1e-9),
                (df['side'] == 'BUY') & (df['prev_position'] > 1e-9) & (df['position_change'] > 1e-9),
                (df['side'] == 'SELL') & (df['position'] <= 1e-9) & (df['prev_position'] > 1e-9),
                (df['side'] == 'SELL') & (df['position_change'] < -1e-9),
            ],
            ['open_buy', 'add_buy', 'close_sell', 'reduce_sell'],
            default='other',
        )
        chunks.append(df)

    if not chunks:
        empty_events = pd.DataFrame()
        empty_open = pd.DataFrame()
        return empty_events, empty_open

    events = pd.concat(chunks, ignore_index=True).sort_values(['dt', 'wallet', 'condition_id', 'outcome']).reset_index(drop=True)
    market_first_trade = events.groupby('condition_id')['dt'].min().rename('first_selected_trade_dt')
    open_buys = events[events['event_type'] == 'open_buy'].copy()
    open_buys = open_buys.merge(wallet_profiles, on='wallet', how='left')
    open_buys['conviction_ratio'] = open_buys['usdc_amount'] / open_buys['median_open_buy_usdc'].replace({0.0: np.nan})
    open_buys['conviction_ratio'] = open_buys['conviction_ratio'].replace([np.inf, -np.inf], np.nan).fillna(1.0)
    open_buys = open_buys.merge(market_first_trade, on='condition_id', how='left')
    open_buys['hours_since_first_selected_trade'] = ((open_buys['dt'] - open_buys['first_selected_trade_dt']).dt.total_seconds() / 3600.0).clip(lower=0.0)
    open_buys['prob_edge'] = open_buys['final_price'] - open_buys['price']
    open_buys['copy_roi'] = np.where(open_buys['token_winner'], 1.0 / open_buys['price'].clip(lower=0.001) - 1.0, -1.0)
    open_buys['copy_roi_capped'] = np.clip(open_buys['copy_roi'], -1.0, 10.0)
    open_buys['price_bucket'] = pd.cut(open_buys['price'], bins=PRICE_BUCKET_BINS, labels=PRICE_BUCKET_LABELS, include_lowest=True).astype(str)
    open_buys = attach_consensus_features(open_buys)
    return events, open_buys


In [218]:
def verify_partial_fill_grouping(dataset, batch_size=300_000):
    columns = ['wallet', 'condition_id', 'outcome', 'dt', 'side', 'quantity', 'price', 'usdc_amount', 'position']
    sample = None
    for batch in dataset.to_batches(columns=columns, batch_size=batch_size):
        df = batch.to_pandas()
        if not df.empty:
            sample = df
            break
    if sample is None or sample.empty:
        return pd.DataFrame({'metric': ['note'], 'value': ['no rows available']})
    sample['dt'] = pd.to_datetime(sample['dt'], utc=True)
    keys = ['wallet', 'condition_id', 'outcome', 'dt', 'side']
    grouped = aggregate_wallet_trades(sample)
    raw_dupe_rows = int(sample.duplicated(keys, keep=False).sum())
    grouped_dupe_rows = int(grouped.duplicated(keys, keep=False).sum())

    raw_signed_qty = np.where(sample['side'] == 'BUY', sample['quantity'], -sample['quantity'])
    raw_prev_pos = sample['position'] - raw_signed_qty
    raw_open_buys = int(((sample['side'] == 'BUY') & (raw_prev_pos <= 1e-9)).sum())
    grouped_open_buys = int(((grouped['side'] == 'BUY') & (grouped['prev_position'] <= 1e-9)).sum())

    report = pd.DataFrame([
        {'metric': 'raw_rows', 'value': int(len(sample))},
        {'metric': 'grouped_rows', 'value': int(len(grouped))},
        {'metric': 'raw_duplicate_key_rows', 'value': raw_dupe_rows},
        {'metric': 'grouped_duplicate_key_rows', 'value': grouped_dupe_rows},
        {'metric': 'raw_open_buy_events', 'value': raw_open_buys},
        {'metric': 'grouped_open_buy_events', 'value': grouped_open_buys},
    ])
    return report

partial_fill_grouping_check = verify_partial_fill_grouping(dataset)
partial_fill_grouping_check


,metric,value
0,raw_rows,300000
1,grouped_rows,222805
2,raw_duplicate_key_rows,112254
3,grouped_duplicate_key_rows,0
4,raw_open_buy_events,42987
5,grouped_open_buy_events,42972


In [219]:
selected_wallets_a = select_wallets(
    train_a_metrics.assign(distinct_markets=9999, recent_open_buy_trades=train_a_metrics['open_buy_trades']),
    BEST_SELECTION_METRIC,
    BEST_TOP_N,
    min_open_buys=10,
    min_volume=250.0,
    min_markets=0,
    min_recent_trades=0,
)
selected_wallet_profiles_a = build_wallet_profiles(dataset, selected_wallets_a, period='train_a')

if CALIBRATION_SIGNALS_PATH.exists():
    train_b_open_buys = pd.read_parquet(CALIBRATION_SIGNALS_PATH)
else:
    _, train_b_open_buys = build_signal_events(dataset, selected_wallet_profiles_a, period='train_b')
    train_b_open_buys.to_parquet(CALIBRATION_SIGNALS_PATH, index=False)

train_b_open_buys[['wallet', 'dt', 'condition_id', 'outcome', 'price', 'conviction_ratio', 'prior_same_24h', 'prior_opp_24h', 'prob_edge']].head()


,wallet,dt,condition_id,outcome,price,conviction_ratio,prior_same_24h,prior_opp_24h,prob_edge
0,0x03626d34381b0337387b0e8464c898f772009661,2025-07-23 00:26:37+00:00,0x62e3be7fefc94de32f0029675f87a73f7794eb903c3a28723410bfd6993b6959,Yes,0.95536,2.407636,0,0,0.04464
1,0x2cde80346eed526242e266476cfcc65073635b98,2025-07-23 02:05:57+00:00,0x69f218208ee7bee0897b5f683790ab945b7db5311886fd39641d476dffdd8944,Yes,0.05000,1.718606,0,0,0.95000
2,0xa8cce5524b0c9664300002f4d1f779c41c3b728f,2025-07-23 15:22:59+00:00,0xef59e051b273dd220dbf62a8d949fe080678cb8405311edee5b0805410400854,Royals,0.39000,0.829787,0,0,0.61000
3,0x43c3a4fb52204a3cdbf1f0042efc584ee7bf09b8,2025-07-23 15:23:07+00:00,0xef59e051b273dd220dbf62a8d949fe080678cb8405311edee5b0805410400854,Royals,0.39000,0.847826,1,0,0.61000
4,0x43c3a4fb52204a3cdbf1f0042efc584ee7bf09b8,2025-07-23 15:33:57+00:00,0x5a3580d653708fbdb171e1e651da735210ce1a3e5ccdf3e7408f56545afa6c68,Athletics,0.44000,0.956522,0,0,-0.44000


## 5. Continuous signal score calibration

In [220]:
def _scale_lookup(series):
    s = series.copy()
    if s.empty:
        return s
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or pd.isna(hi) or hi - lo < 1e-12:
        return pd.Series(0.5, index=s.index)
    return ((s - lo) / (hi - lo)).clip(0.0, 1.0)


def build_calibration_tables(calibration_signals, price_prior=50, consensus_prior=25):
    global_edge = calibration_signals['prob_edge'].mean()
    price_table = calibration_signals.groupby('price_bucket').agg(n=('wallet', 'size'), sum_edge=('prob_edge', 'sum'))
    price_table['smoothed_edge'] = (price_table['sum_edge'] + price_prior * global_edge) / (price_table['n'] + price_prior)
    price_table['price_score'] = _scale_lookup(price_table['smoothed_edge'])
    consensus_df = calibration_signals.copy()
    consensus_df['same_bucket'] = consensus_df['prior_same_24h'].clip(upper=3)
    consensus_df['opp_bucket'] = consensus_df['prior_opp_24h'].clip(upper=2)
    consensus_table = consensus_df.groupby(['same_bucket', 'opp_bucket']).agg(n=('wallet', 'size'), sum_edge=('prob_edge', 'sum'))
    consensus_table['smoothed_edge'] = (consensus_table['sum_edge'] + consensus_prior * global_edge) / (consensus_table['n'] + consensus_prior)
    consensus_table['consensus_score'] = _scale_lookup(consensus_table['smoothed_edge'])
    return price_table.reset_index(), consensus_table.reset_index(), global_edge


def apply_signal_score(signals, price_table, consensus_table):
    price_map = price_table.set_index('price_bucket')['price_score']
    consensus_map = consensus_table.set_index(['same_bucket', 'opp_bucket'])['consensus_score']
    scored = signals.copy()
    scored['same_bucket'] = scored['prior_same_24h'].clip(upper=3)
    scored['opp_bucket'] = scored['prior_opp_24h'].clip(upper=2)
    scored['wallet_component'] = scored['wallet_quality'].fillna(0.0)
    scored['conviction_component'] = np.clip(np.log1p(scored['conviction_ratio']) / np.log(3.0), 0.0, 1.5).clip(0.0, 1.0)
    scored['price_component'] = scored['price_bucket'].map(price_map).fillna(price_map.mean())
    scored['consensus_component'] = list(zip(scored['same_bucket'], scored['opp_bucket']))
    scored['consensus_component'] = scored['consensus_component'].map(consensus_map).fillna(consensus_map.mean())
    scored['signal_score'] = (
        0.45 * scored['wallet_component']
        + 0.15 * scored['conviction_component']
        + 0.20 * scored['price_component']
        + 0.20 * scored['consensus_component']
    )
    return scored

price_table, consensus_table, global_edge = build_calibration_tables(train_b_open_buys)
train_b_scored = apply_signal_score(train_b_open_buys, price_table, consensus_table)
train_b_scored[['signal_score', 'wallet_component', 'conviction_component', 'price_component', 'consensus_component']].describe()


,signal_score,wallet_component,conviction_component,price_component,consensus_component
count,2188.000000,2188.000000,2.188000e+03,2188.000000,2188.000000
mean,0.680098,0.634936,5.848070e-01,0.752927,0.780351
std,0.144743,0.276360,3.996761e-01,0.261594,0.263445
min,0.205368,0.020000,1.432097e-17,0.000000,0.000000
25%,0.573962,0.500000,1.667506e-01,0.698473,0.881645
50%,0.678320,0.680000,6.554384e-01,0.708699,0.881645
75%,0.790024,0.960000,1.000000e+00,1.000000,0.881645
max,0.967329,1.000000,1.000000e+00,1.000000,1.000000


In [221]:
score_deciles = train_b_scored.assign(score_decile=pd.qcut(train_b_scored['signal_score'].rank(method='first'), 10, labels=False))
score_deciles.groupby('score_decile').agg(
    signals=('wallet', 'size'),
    avg_prob_edge=('prob_edge', 'mean'),
    avg_copy_roi_capped=('copy_roi_capped', 'mean'),
    win_rate=('token_winner', 'mean'),
    avg_price=('price', 'mean'),
)


,signals,avg_prob_edge,avg_copy_roi_capped,win_rate,avg_price
score_decile,,,,,
0,219,0.230889,1.860643,0.474886,0.243997
1,219,0.234513,1.754792,0.447489,0.212976
2,219,0.157980,1.051924,0.484018,0.326038
3,218,0.058839,0.362047,0.376147,0.317308
4,219,0.112000,0.493976,0.529680,0.417681
5,219,0.122451,0.589540,0.429224,0.306773
6,218,0.095610,0.421609,0.467890,0.372280
7,219,0.142978,0.811944,0.447489,0.304511
8,219,0.213770,1.636608,0.342466,0.128696


In [222]:
threshold_grid = np.round(np.arange(0.50, 0.96, 0.05), 2)
threshold_rows = []
for threshold in threshold_grid:
    subset = train_b_scored[train_b_scored['signal_score'] >= threshold]
    threshold_rows.append({
        'threshold': threshold,
        'signals': len(subset),
        'avg_prob_edge': subset['prob_edge'].mean() if len(subset) else np.nan,
        'avg_copy_roi_capped': subset['copy_roi_capped'].mean() if len(subset) else np.nan,
        'win_rate': subset['token_winner'].mean() if len(subset) else np.nan,
    })
threshold_summary = pd.DataFrame(threshold_rows)
threshold_summary


,threshold,signals,avg_prob_edge,avg_copy_roi_capped,win_rate
0,0.50,1855,0.124146,0.676040,0.407547
1,0.55,1723,0.122598,0.671517,0.406268
2,0.60,1583,0.120652,0.659474,0.397347
3,0.65,1293,0.130928,0.683591,0.401392
4,0.70,959,0.128647,0.683786,0.361835
5,0.75,680,0.141320,0.849165,0.323529
6,0.80,464,0.153402,0.907843,0.271552
7,0.85,290,0.073147,0.113277,0.186207
8,0.90,200,0.083354,0.105830,0.185000
9,0.95,93,-0.005866,-0.645161,0.032258


In [223]:
threshold_candidates = threshold_summary[threshold_summary['signals'] >= 30].copy()
if threshold_candidates.empty:
    SIGNAL_THRESHOLD = float(threshold_summary.sort_values('avg_copy_roi_capped', ascending=False).iloc[0]['threshold'])
else:
    SIGNAL_THRESHOLD = float(threshold_candidates.sort_values(['avg_copy_roi_capped', 'avg_prob_edge'], ascending=False).iloc[0]['threshold'])
SIGNAL_THRESHOLD


0.8

## 6. Test-set benchmark comparison and trade ledger

This section is benchmark-first.

All strategies below use the same held-out test data, the same forward-fill execution model, the same 10 minute fill horizon, the same slippage / fee assumptions, the same one-trade-per-market dedupe rule, and the same daily cap.

That makes the comparison fair: the only difference is the **trigger rule** and whether sizing is fixed or score-scaled.


In [224]:
from polymarket_analysis.backtest.execution_tape import (
    build_execution_tape,
    build_tape_groups,
    normalize_execution_tape,
    attach_forward_fills,
)

def compute_simple_kelly_fraction(trades, kelly_scale=KELLY_SCALE, max_fraction=KELLY_MAX_FRACTION):
    if trades.empty:
        return pd.Series(dtype=float)
    score = trades['signal_score'].fillna(0.5).clip(0.0, 1.0)
    p_hat = (0.5 + 0.35 * (score - 0.5)).clip(0.05, 0.95)
    price = trades['price'].clip(lower=0.01, upper=0.99)
    b = (1.0 - price) / price
    full_kelly = ((b * p_hat - (1.0 - p_hat)) / b).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return (full_kelly * kelly_scale).clip(lower=0.0, upper=max_fraction)


def backtest_strategy(signals, mask, tape_groups, strategy_name, trigger_rule, base_stake_usdc=100.0, dynamic_sizing=False, latency_seconds=0, fill_horizon_seconds=600, slippage_bps=50.0, fee_bps=10.0, max_signals_per_day=20, dedupe_by_market=True, starting_capital=10000.0, cohort_name='default', min_stake_usdc=KELLY_MIN_STAKE_USDC, max_stake_usdc=KELLY_MAX_STAKE_USDC, max_rel_price_diff_by_bucket=None, max_price_premium=0.0):
    trades = signals[mask].copy().sort_values('dt')
    trades['strategy'] = strategy_name
    trades['trigger_rule'] = trigger_rule
    trades['cohort'] = cohort_name
    trades['sizing_mode'] = 'kelly_simple' if dynamic_sizing else 'fixed'
    if dedupe_by_market:
        trades = trades.drop_duplicates('market_key', keep='first')
    trades['trade_date'] = trades['dt'].dt.floor('D')
    if max_signals_per_day is not None:
        trades['daily_rank'] = trades.groupby('trade_date').cumcount() + 1
        trades = trades[trades['daily_rank'] <= max_signals_per_day].copy()
    if trades.empty:
        empty_daily = pd.DataFrame(columns=['trade_date', 'trades', 'net_pnl_usdc', 'cum_net_pnl_usdc', 'equity_usdc'])
        return trades, empty_daily, trades, empty_daily.copy()

    trades = trades.reset_index(drop=True)
    trades['trigger_id'] = np.arange(len(trades), dtype=int)

    if dynamic_sizing:
        trades['kelly_fraction'] = compute_simple_kelly_fraction(trades)
        raw_stake = starting_capital * trades['kelly_fraction']
        trades['stake_usdc'] = np.where(
            trades['kelly_fraction'] > 1e-12,
            np.clip(raw_stake, min_stake_usdc, max_stake_usdc),
            0.0,
        )
        trades = trades[trades['stake_usdc'] > 0.0].copy()
    else:
        trades['kelly_fraction'] = np.nan
        trades['stake_usdc'] = float(base_stake_usdc)
    if trades.empty:
        empty_daily = pd.DataFrame(columns=['trade_date', 'trades', 'net_pnl_usdc', 'cum_net_pnl_usdc', 'equity_usdc'])
        return trades, empty_daily, trades, empty_daily.copy()

    candidate_triggers = trades.copy()
    fee = fee_bps / 10000.0
    candidate_triggers['trigger_gross_roi'] = np.where(candidate_triggers['token_winner'], 1.0 / candidate_triggers['price'].clip(lower=0.001) - 1.0, -1.0)
    candidate_triggers['trigger_net_roi'] = candidate_triggers['trigger_gross_roi'] - fee
    if 'trade_pnl' in candidate_triggers.columns:
        candidate_triggers['trigger_net_pnl_usdc'] = candidate_triggers['trade_pnl'].astype(float)
    else:
        candidate_triggers['trigger_net_pnl_usdc'] = candidate_triggers['stake_usdc'] * candidate_triggers['trigger_net_roi']
    theoretical_daily = (
        candidate_triggers.groupby('trade_date')
        .agg(trades=('market_key', 'size'), net_pnl_usdc=('trigger_net_pnl_usdc', 'sum'))
        .reset_index()
        .sort_values('trade_date')
    )
    theoretical_daily['cum_net_pnl_usdc'] = theoretical_daily['net_pnl_usdc'].cumsum()
    theoretical_daily['equity_usdc'] = starting_capital + theoretical_daily['cum_net_pnl_usdc']

    trades = attach_forward_fills(
        trades,
        tape_groups=tape_groups,
        latency_seconds=latency_seconds,
        fill_horizon_seconds=fill_horizon_seconds,
        slippage_bps=slippage_bps,
        min_fill_ratio=1.0,
        max_price_premium=max_price_premium,
        max_rel_price_diff_by_bucket=max_rel_price_diff_by_bucket,
    )
    if trades.empty:
        empty_daily = pd.DataFrame(columns=['trade_date', 'trades', 'net_pnl_usdc', 'cum_net_pnl_usdc', 'equity_usdc'])
        unfilled = candidate_triggers.copy()
        unfilled['fill_status'] = 'no_fill'
        return trades, empty_daily, unfilled, theoretical_daily

    filled_ids = set(trades['trigger_id'].astype(int).tolist())
    unfilled = candidate_triggers[~candidate_triggers['trigger_id'].isin(filled_ids)].copy()
    if not unfilled.empty:
        unfilled['fill_status'] = 'no_fill'

    trades['gross_roi'] = np.where(trades['token_winner'], 1.0 / trades['exec_price'] - 1.0, -1.0)
    trades['net_roi'] = trades['gross_roi'] - fee
    trades['net_pnl_usdc'] = trades['filled_usdc'] * trades['net_roi']
    trades['fill_delay_minutes'] = (trades['tape_dt'] - trades['dt']).dt.total_seconds() / 60.0
    daily = (
        trades.groupby('trade_date')
        .agg(trades=('market_key', 'size'), net_pnl_usdc=('net_pnl_usdc', 'sum'))
        .reset_index()
        .sort_values('trade_date')
    )
    daily['cum_net_pnl_usdc'] = daily['net_pnl_usdc'].cumsum()
    daily['equity_usdc'] = starting_capital + daily['cum_net_pnl_usdc']
    return trades, daily, unfilled, theoretical_daily


def summarize_backtest(trades, daily):
    if trades.empty:
        return pd.Series({'filled_trades': 0, 'net_pnl_usdc': 0.0, 'net_roi_on_stake': np.nan, 'win_rate': np.nan})
    total_stake = float(trades['filled_usdc'].sum())
    max_drawdown = 0.0
    if not daily.empty:
        running_peak = daily['equity_usdc'].cummax()
        max_drawdown = float((running_peak - daily['equity_usdc']).max())
    return pd.Series({
        'filled_trades': int(len(trades)),
        'net_pnl_usdc': float(trades['net_pnl_usdc'].sum()),
        'net_roi_on_stake': float(trades['net_pnl_usdc'].sum() / total_stake) if total_stake > 0 else np.nan,
        'win_rate': float(trades['token_winner'].mean()),
        'avg_signal_score': float(trades['signal_score'].mean()) if 'signal_score' in trades.columns else np.nan,
        'avg_kelly_fraction': float(trades['kelly_fraction'].mean()) if 'kelly_fraction' in trades.columns else np.nan,
        'avg_fill_delay_minutes': float(trades['fill_delay_minutes'].mean()),
        'opposite_fill_share': float(trades['opposite_fill_share'].mean()),
        'max_drawdown_usdc': max_drawdown,
    })


def build_trigger_ledger(filled_trades, unfilled_triggers):
    ledger_columns = [
        'cohort', 'strategy', 'trigger_rule', 'sizing_mode', 'fill_status', 'trigger_id', 'dt', 'trade_date', 'tape_dt', 'fill_delay_minutes',
        'trigger_tx_hash', 'fill_tx_hash',
        'wallet', 'condition_id', 'outcome', 'market_key',
        'price', 'quantity', 'exec_price', 'stake_usdc', 'filled_usdc', 'fill_ratio', 'kelly_fraction',
        'wallet_quality', 'signal_score', 'wallet_component', 'conviction_component', 'price_component', 'consensus_component',
        'conviction_ratio', 'prior_same_any', 'prior_opp_any', 'prior_same_24h', 'prior_opp_24h', 'price_bucket',
        'token_winner', 'prob_edge', 'gross_roi', 'net_roi', 'net_pnl_usdc',
        'trigger_gross_roi', 'trigger_net_roi', 'trigger_net_pnl_usdc',
    ]

    parts = []
    if filled_trades is not None and not filled_trades.empty:
        filled = filled_trades.copy()
        filled['fill_status'] = 'filled'
        parts.append(filled)
    if unfilled_triggers is not None and not unfilled_triggers.empty:
        unfilled = unfilled_triggers.copy()
        unfilled['fill_status'] = 'no_fill'
        if 'filled_usdc' not in unfilled.columns:
            unfilled['filled_usdc'] = 0.0
        if 'fill_ratio' not in unfilled.columns:
            unfilled['fill_ratio'] = 0.0
        if 'exec_price' not in unfilled.columns:
            unfilled['exec_price'] = np.nan
        if 'fill_tx_hash' not in unfilled.columns:
            unfilled['fill_tx_hash'] = np.nan
        if 'tape_dt' not in unfilled.columns:
            unfilled['tape_dt'] = pd.NaT
        if 'fill_delay_minutes' not in unfilled.columns:
            unfilled['fill_delay_minutes'] = np.nan
        parts.append(unfilled)

    if not parts:
        return pd.DataFrame(columns=ledger_columns)

    ledger = pd.concat(parts, ignore_index=True, sort=False)
    for col in ['trigger_tx_hash', 'fill_tx_hash']:
        if col not in ledger.columns:
            ledger[col] = np.nan
    available = [c for c in ledger_columns if c in ledger.columns]
    return ledger[available].sort_values(['dt', 'strategy', 'trigger_id']).reset_index(drop=True)


In [225]:
if CALIBRATION_SIGNALS_PATH.exists():
    train_b_open_buys = pd.read_parquet(CALIBRATION_SIGNALS_PATH)


def _with_wallet_quality(df, score_col, ascending=False, top_n=100):
    ranked = df.sort_values(score_col, ascending=ascending).head(top_n).copy().reset_index(drop=True)
    if ranked.empty:
        ranked['wallet_quality'] = []
        return ranked[['wallet', 'wallet_quality']]
    rank_values = ranked[score_col].rank(method='first', ascending=ascending, pct=True)
    ranked['wallet_quality'] = rank_values
    return ranked[['wallet', 'wallet_quality']]


def build_wallet_cohorts(full_train_metrics, train_b_open_buys, base_selected_wallets, top_n=COHORT_TOP_N_DEFAULT):
    cohorts = {}
    top_n = int(top_n)
    core = base_selected_wallets[['wallet', 'wallet_quality']].copy().reset_index(drop=True)
    cohorts['quality_core'] = core

    eligible = full_train_metrics[
        (full_train_metrics['open_buy_trades'] >= 20)
        & (full_train_metrics['volume'] >= 500.0)
    ][['wallet', 'prob_edge_shrunk', 'total_copy_pnl_usdc', 'copy_roi_std']].copy()

    early_stats = (
        train_b_open_buys.groupby('wallet')
        .agg(
            median_hours_since_first=('hours_since_first_selected_trade', 'median'),
            train_b_open_buys=('wallet', 'size'),
        )
        .reset_index()
    )
    early = eligible.merge(early_stats, on='wallet', how='left')
    early = early[(early['train_b_open_buys'] >= 5) & early['median_hours_since_first'].notna()].copy()
    early['early_score'] = early['prob_edge_shrunk'].fillna(0.0) - 0.03 * early['median_hours_since_first']
    cohorts['early_entry'] = _with_wallet_quality(early[['wallet', 'early_score']], 'early_score', ascending=False, top_n=top_n)

    smooth = eligible.copy()
    smooth['copy_roi_std'] = smooth['copy_roi_std'].fillna(smooth['copy_roi_std'].median())
    smooth = smooth[smooth['total_copy_pnl_usdc'] > 0].copy()
    smooth['smooth_score'] = smooth['total_copy_pnl_usdc'] / (smooth['copy_roi_std'].clip(lower=0.02))
    cohorts['smooth_pnl'] = _with_wallet_quality(smooth[['wallet', 'smooth_score']], 'smooth_score', ascending=False, top_n=top_n)

    return {name: df.drop_duplicates('wallet').reset_index(drop=True) for name, df in cohorts.items() if not df.empty}


cohort_top_n = BEST_TOP_N if 'BEST_TOP_N' in globals() else COHORT_TOP_N_DEFAULT
wallet_cohorts = build_wallet_cohorts(full_train_metrics, train_b_open_buys, selected_wallets, top_n=cohort_top_n)

cohort_profiles = {}
cohort_test_signals = {}
cohort_train_ref_signals = {}
for cohort_name, cohort_wallets in wallet_cohorts.items():
    profiles = build_wallet_profiles(dataset, cohort_wallets, period='full_train')
    _, cohort_test_open_buys = build_signal_events(dataset, profiles, period='test')
    _, cohort_train_open_buys = build_signal_events(dataset, profiles, period='train_b')
    cohort_test_scored = apply_signal_score(cohort_test_open_buys, price_table, consensus_table)
    cohort_train_scored = apply_signal_score(cohort_train_open_buys, price_table, consensus_table)
    cohort_profiles[cohort_name] = profiles
    cohort_test_signals[cohort_name] = cohort_test_scored
    cohort_train_ref_signals[cohort_name] = cohort_train_scored

all_test_condition_ids = pd.Index([])
if cohort_test_signals:
    all_test_condition_ids = pd.Index(pd.concat([df['condition_id'] for df in cohort_test_signals.values() if not df.empty], ignore_index=True).drop_duplicates())
all_train_condition_ids = pd.Index([])
if cohort_train_ref_signals:
    all_train_condition_ids = pd.Index(pd.concat([df['condition_id'] for df in cohort_train_ref_signals.values() if not df.empty], ignore_index=True).drop_duplicates())

if EXEC_TAPE_TEST_PATH.exists():
    execution_tape_test = pd.read_parquet(EXEC_TAPE_TEST_PATH)
else:
    execution_tape_test = build_execution_tape(dataset, all_test_condition_ids.tolist())
    execution_tape_test.to_parquet(EXEC_TAPE_TEST_PATH, index=False)

if EXEC_TAPE_TRAIN_B_PATH.exists():
    execution_tape_train_b = pd.read_parquet(EXEC_TAPE_TRAIN_B_PATH)
else:
    execution_tape_train_b = build_execution_tape(
        dataset,
        all_train_condition_ids.tolist(),
        start_date=TRAIN_A_END_DATE + datetime.timedelta(days=1),
        end_date=END_DATE_TRAIN,
    )
    execution_tape_train_b.to_parquet(EXEC_TAPE_TRAIN_B_PATH, index=False)

execution_tape_test['tape_dt'] = pd.to_datetime(execution_tape_test['tape_dt'], utc=True)
execution_tape_train_b['tape_dt'] = pd.to_datetime(execution_tape_train_b['tape_dt'], utc=True)
tape_groups = build_tape_groups(normalize_execution_tape(execution_tape_test))
tape_groups_train_ref = build_tape_groups(normalize_execution_tape(execution_tape_train_b))

strategy_specs = []
for cohort_name, scored_df in cohort_test_signals.items():
    if scored_df.empty:
        continue
    strategy_specs.extend([
        {
            'name': f'{cohort_name}__score_threshold_{SIGNAL_THRESHOLD:.2f}',
            'cohort': cohort_name,
            'signals': scored_df,
            'mask': scored_df['signal_score'] >= SIGNAL_THRESHOLD,
            'trigger_rule': f'signal_score >= {SIGNAL_THRESHOLD:.2f}',
            'dynamic_sizing': True,
        },
        {
            'name': f'{cohort_name}__all_open_buys',
            'cohort': cohort_name,
            'signals': scored_df,
            'mask': scored_df['wallet'].notna(),
            'trigger_rule': 'all selected-wallet open_buy events',
            'dynamic_sizing': False,
        },
    ])

strategy_specs_train_ref = []
for cohort_name, scored_df in cohort_train_ref_signals.items():
    if scored_df.empty:
        continue
    strategy_specs_train_ref.extend([
        {
            'name': f'{cohort_name}__score_threshold_{SIGNAL_THRESHOLD:.2f}',
            'cohort': cohort_name,
            'signals': scored_df,
            'mask': scored_df['signal_score'] >= SIGNAL_THRESHOLD,
            'trigger_rule': f'signal_score >= {SIGNAL_THRESHOLD:.2f}',
            'dynamic_sizing': True,
        },
        {
            'name': f'{cohort_name}__all_open_buys',
            'cohort': cohort_name,
            'signals': scored_df,
            'mask': scored_df['wallet'].notna(),
            'trigger_rule': 'all selected-wallet open_buy events',
            'dynamic_sizing': False,
        },
    ])

strategy_runs = {}
strategy_rows = []
for spec in strategy_specs:
    trades_bt, daily_bt, unfilled_bt, theoretical_daily_bt = backtest_strategy(
        spec['signals'],
        mask=spec['mask'],
        tape_groups=tape_groups,
        strategy_name=spec['name'],
        trigger_rule=spec['trigger_rule'],
        base_stake_usdc=100.0,
        dynamic_sizing=spec['dynamic_sizing'],
        latency_seconds=0,
        fill_horizon_seconds=600,
        slippage_bps=50.0,
        fee_bps=10.0,
        max_signals_per_day=20,
        dedupe_by_market=True,
        starting_capital=10000.0,
        cohort_name=spec['cohort'],
    )
    strategy_runs[spec['name']] = {'trades': trades_bt, 'daily': daily_bt, 'unfilled_triggers': unfilled_bt, 'theoretical_daily': theoretical_daily_bt, 'spec': spec}
    row = summarize_backtest(trades_bt, daily_bt).to_dict()
    row['strategy'] = spec['name']
    row['cohort'] = spec['cohort']
    row['trigger_rule'] = spec['trigger_rule']
    row['dynamic_sizing'] = spec['dynamic_sizing']
    row['triggered_signals'] = int(len(trades_bt) + len(unfilled_bt))
    row['unfilled_triggers'] = int(len(unfilled_bt))
    row['fill_rate'] = float(len(trades_bt) / (len(trades_bt) + len(unfilled_bt))) if (len(trades_bt) + len(unfilled_bt)) else np.nan
    strategy_rows.append(row)

strategy_runs_train_ref = {}
strategy_rows_train_ref = []
for spec in strategy_specs_train_ref:
    trades_bt, daily_bt, unfilled_bt, theoretical_daily_bt = backtest_strategy(
        spec['signals'],
        mask=spec['mask'],
        tape_groups=tape_groups_train_ref,
        strategy_name=spec['name'],
        trigger_rule=spec['trigger_rule'],
        base_stake_usdc=100.0,
        dynamic_sizing=spec['dynamic_sizing'],
        latency_seconds=0,
        fill_horizon_seconds=600,
        slippage_bps=50.0,
        fee_bps=10.0,
        max_signals_per_day=20,
        dedupe_by_market=True,
        starting_capital=10000.0,
        cohort_name=spec['cohort'],
    )
    strategy_runs_train_ref[spec['name']] = {'trades': trades_bt, 'daily': daily_bt, 'unfilled_triggers': unfilled_bt, 'theoretical_daily': theoretical_daily_bt, 'spec': spec}
    row = summarize_backtest(trades_bt, daily_bt).to_dict()
    row['strategy'] = spec['name']
    row['cohort'] = spec['cohort']
    row['trigger_rule'] = spec['trigger_rule']
    row['dynamic_sizing'] = spec['dynamic_sizing']
    row['triggered_signals'] = int(len(trades_bt) + len(unfilled_bt))
    row['unfilled_triggers'] = int(len(unfilled_bt))
    row['fill_rate'] = float(len(trades_bt) / (len(trades_bt) + len(unfilled_bt))) if (len(trades_bt) + len(unfilled_bt)) else np.nan
    strategy_rows_train_ref.append(row)

strategy_summary = pd.DataFrame(strategy_rows).set_index('strategy').sort_values('net_roi_on_stake', ascending=False)
strategy_summary_train_ref = pd.DataFrame(strategy_rows_train_ref).set_index('strategy').sort_values('net_roi_on_stake', ascending=False)

trigger_ledgers = {name: build_trigger_ledger(run['trades'], run['unfilled_triggers']) for name, run in strategy_runs.items()}
all_trigger_ledger = pd.concat([ledger for ledger in trigger_ledgers.values() if not ledger.empty], ignore_index=True) if any(not ledger.empty for ledger in trigger_ledgers.values()) else pd.DataFrame()

SCORE_STRATEGY_NAME = f'quality_core__score_threshold_{SIGNAL_THRESHOLD:.2f}'
test_scored = cohort_test_signals.get('quality_core', pd.DataFrame())
test_trades_bt = strategy_runs[SCORE_STRATEGY_NAME]['trades'] if SCORE_STRATEGY_NAME in strategy_runs else pd.DataFrame()
test_daily_bt = strategy_runs[SCORE_STRATEGY_NAME]['daily'] if SCORE_STRATEGY_NAME in strategy_runs else pd.DataFrame()
test_theoretical_daily_bt = strategy_runs[SCORE_STRATEGY_NAME]['theoretical_daily'] if SCORE_STRATEGY_NAME in strategy_runs else pd.DataFrame()
score_trigger_ledger = trigger_ledgers[SCORE_STRATEGY_NAME] if SCORE_STRATEGY_NAME in trigger_ledgers else pd.DataFrame()
score_unfilled_trigger_ledger = score_trigger_ledger[score_trigger_ledger['fill_status'] == 'no_fill'].copy() if not score_trigger_ledger.empty else pd.DataFrame()

strategy_summary


,filled_trades,net_pnl_usdc,net_roi_on_stake,win_rate,avg_signal_score,avg_kelly_fraction,avg_fill_delay_minutes,opposite_fill_share,max_drawdown_usdc,cohort,trigger_rule,dynamic_sizing,triggered_signals,unfilled_triggers,fill_rate
strategy,,,,,,,,,,,,,,,
early_entry__all_open_buys,13.0,826.448670,0.635730,0.538462,0.655732,NaN,3.871795,0.268154,398.972007,early_entry,all selected-wallet open_buy events,False,98,85,0.132653
quality_core__all_open_buys,30.0,1346.751759,0.448917,0.300000,0.659586,NaN,3.292222,0.318366,700.700000,quality_core,all selected-wallet open_buy events,False,580,550,0.051724
smooth_pnl__all_open_buys,120.0,-1493.392720,-0.124449,0.458333,0.647512,NaN,3.830833,0.224357,1137.175196,smooth_pnl,all selected-wallet open_buy events,False,580,460,0.206897
smooth_pnl__score_threshold_0.80,105.0,-9723.833431,-0.184862,0.400000,0.820158,0.052944,4.117460,0.217490,13042.441924,smooth_pnl,signal_score >= 0.80,True,556,451,0.188849
quality_core__score_threshold_0.80,1.0,-750.750000,-1.001000,0.000000,0.882069,0.100000,9.900000,0.012663,0.000000,quality_core,signal_score >= 0.80,True,530,529,0.001887
early_entry__score_threshold_0.80,1.0,-750.750000,-1.001000,0.000000,0.892069,0.093631,7.166667,0.000000,0.000000,early_entry,signal_score >= 0.80,True,9,8,0.111111


In [226]:
# trigger_ledgers = {name: build_trigger_ledger(run['trades'], run['unfilled_triggers']) for name, run in strategy_runs.items()}
# pd.set_option('display.max_columns', None)
# trigger_ledgers['quality_core__all_open_buys']

### Trigger trade ledger

Each row below is a trigger event after all rules were applied (filled and unfilled):
- the trigger rule fired
- only the first qualifying trade per market was kept
- the daily cap was applied
- the trade was forward-filled against the execution tape

The table includes trigger parameters (`signal_score`, `wallet_quality`, consensus counts, conviction, price bucket), trigger transaction hash, fill transaction hash (when filled), and realized/theoretical PnL fields.


In [227]:
strategy_parameters = pd.DataFrame([
    {
        'strategy': name,
        'cohort': run['spec']['cohort'],
        'trigger_rule': run['spec']['trigger_rule'],
        'dynamic_sizing': run['spec']['dynamic_sizing'],
        'filled_trades': len(run['trades']),
        'unfilled_triggers': len(run['unfilled_triggers']),
        'triggered_signals': len(run['trades']) + len(run['unfilled_triggers']),
    }
    for name, run in strategy_runs.items()
]).sort_values(['cohort', 'triggered_signals', 'strategy'], ascending=[True, False, True])

strategy_parameters['fill_rate'] = np.where(
    strategy_parameters['triggered_signals'] > 0,
    strategy_parameters['filled_trades'] / strategy_parameters['triggered_signals'],
    np.nan,
)

strategy_parameters


,strategy,cohort,trigger_rule,dynamic_sizing,filled_trades,unfilled_triggers,triggered_signals,fill_rate
3,early_entry__all_open_buys,early_entry,all selected-wallet open_buy events,False,13,85,98,0.132653
2,early_entry__score_threshold_0.80,early_entry,signal_score >= 0.80,True,1,8,9,0.111111
1,quality_core__all_open_buys,quality_core,all selected-wallet open_buy events,False,30,550,580,0.051724
0,quality_core__score_threshold_0.80,quality_core,signal_score >= 0.80,True,1,529,530,0.001887
5,smooth_pnl__all_open_buys,smooth_pnl,all selected-wallet open_buy events,False,120,460,580,0.206897
4,smooth_pnl__score_threshold_0.80,smooth_pnl,signal_score >= 0.80,True,105,451,556,0.188849


In [228]:
all_trigger_ledger['net_pnl_usdc'][2]

np.float64(nan)

In [229]:
print(all_trigger_ledger.iloc[4].to_json(indent=2))

{
  "cohort":"quality_core",
  "strategy":"quality_core__score_threshold_0.80",
  "trigger_rule":"signal_score >= 0.80",
  "sizing_mode":"kelly_simple",
  "fill_status":"filled",
  "trigger_id":4,
  "dt":1770773821000,
  "trade_date":1770768000000,
  "tape_dt":1770774415000,
  "fill_delay_minutes":9.9,
  "trigger_tx_hash":"0x27df9b5abd6d236d117cd623723fba7d66251776403bba05aa496b551de1a0ee",
  "fill_tx_hash":"0xced9f287df4d122693de020a886a4d835b2e4dfe5cdf3b562b5ab41331078ee7",
  "wallet":"0x54a2c4cfc4332d831acc3f5a860d6540982c1d43",
  "condition_id":"0x5365f6a854d4d74b2c60b1fea5ad662c152965412eb8b9179b83081fda54e072",
  "outcome":"Baylor Bears",
  "market_key":"0x5365f6a854d4d74b2c60b1fea5ad662c152965412eb8b9179b83081fda54e072|Baylor Bears",
  "price":0.24,
  "quantity":215.0,
  "exec_price":0.21105,
  "stake_usdc":750.0,
  "filled_usdc":750.0,
  "fill_ratio":1.0,
  "kelly_fraction":0.1,
  "wallet_quality":0.92,
  "signal_score":0.8820687712,
  "wallet_component":0.92,
  "conviction_com

In [230]:
all_trigger_ledger.sort_values(['strategy', 'dt']).reset_index(drop=True)


,cohort,strategy,trigger_rule,sizing_mode,fill_status,trigger_id,dt,trade_date,tape_dt,fill_delay_minutes,trigger_tx_hash,fill_tx_hash,wallet,condition_id,outcome,market_key,price,quantity,exec_price,stake_usdc,filled_usdc,fill_ratio,kelly_fraction,wallet_quality,signal_score,wallet_component,conviction_component,price_component,consensus_component,conviction_ratio,prior_same_any,prior_opp_any,prior_same_24h,prior_opp_24h,price_bucket,token_winner,prob_edge,gross_roi,net_roi,net_pnl_usdc,trigger_gross_roi,trigger_net_roi,trigger_net_pnl_usdc
0,early_entry,early_entry__all_open_buys,all selected-wallet open_buy events,fixed,no_fill,0,2026-02-11 16:29:11+00:00,2026-02-11 00:00:00+00:00,NaT,NaN,0xe8b690244f2ccf4bed1a8eed295f166f76413f66a16a1da4aa0b402e8f64dc6f,NaN,0x90a0774c37e43c03116917adf3563aee1eb67e0a,0xc300271dbf9f46715ee68d54c6ec21e12f920575cb9c8ec28d2fc36683cda1b4,Yes,0xc300271dbf9f46715ee68d54c6ec21e12f920575cb9c8ec28d2fc36683cda1b4|Yes,0.610000,99.340000,NaN,100.000000,0.0,0.0,NaN,0.7500,0.754817,0.7500,1.000000,0.454942,0.881645,2.741964,0,0,0,0,0.60-0.75,True,0.390000,NaN,NaN,NaN,0.639344,0.638344,38.742600
1,early_entry,early_entry__all_open_buys,all selected-wallet open_buy events,fixed,filled,1,2026-02-11 22:05:47+00:00,2026-02-11 00:00:00+00:00,2026-02-11 22:07:49+00:00,2.033333,0x0b4e17b62b0718361f6464e136c9e9f2db3bfdb1c89f79a98dcebb3e19315a21,0xf6e3923b060f322ed6137ac1de98c114e4894614d36a032d567e55321fc83e83,0x526ddfc4a5c40158b9679e8e97db67017dc373f1,0xf23a18c26127d9b153ef1ca40ec94f7603b18170c474d0415cdca71ac52dbebd,Spurs,0xf23a18c26127d9b153ef1ca40ec94f7603b18170c474d0415cdca71ac52dbebd|Spurs,0.670000,29.850745,0.6633,100.000000,100.0,1.0,NaN,1.0000,0.835668,1.0000,0.789005,0.454942,0.881645,1.379310,0,0,0,0,0.60-0.75,True,0.330000,0.507613,0.506613,50.661333,NaN,NaN,NaN
2,early_entry,early_entry__all_open_buys,all selected-wallet open_buy events,fixed,no_fill,2,2026-02-12 03:24:57+00:00,2026-02-12 00:00:00+00:00,NaT,NaN,0xe69f35a323e3a3f07c39d2fb15792f723ce900ff284d7ec260cf67f39067b43e,NaN,0x90a0774c37e43c03116917adf3563aee1eb67e0a,0x4b78de44ea49d0eb8d5fee85352294b6f679a47c56944c6ad5c945e9a2ba7211,Yes,0x4b78de44ea49d0eb8d5fee85352294b6f679a47c56944c6ad5c945e9a2ba7211|Yes,0.810000,371.995788,NaN,100.000000,0.0,0.0,NaN,0.7500,0.752435,0.7500,1.000000,0.443030,0.881645,13.634235,0,0,0,0,0.75-0.90,False,-0.810000,NaN,NaN,NaN,-1.000000,-1.001000,-301.316589
3,early_entry,early_entry__all_open_buys,all selected-wallet open_buy events,fixed,no_fill,3,2026-02-14 09:05:57+00:00,2026-02-14 00:00:00+00:00,NaT,NaN,0xaef8e95424087e67c20c35975234a27cf286a2701e515e44950ad69f38edb610,NaN,0x5bbefc673462f1955e31b4a2347450724946c65d,0x2e80875ed4c846dafced7dd6937e597af26ab94b53d226e9ad86c946c6b1b8e2,Aurora Gaming,0x2e80875ed4c846dafced7dd6937e597af26ab94b53d226e9ad86c946c6b1b8e2|Aurora Gaming,0.010000,559.250000,NaN,100.000000,0.0,0.0,NaN,0.3125,0.666954,0.3125,1.000000,1.000000,0.881645,4.660417,0,0,0,0,0.00-0.10,False,-0.010000,NaN,NaN,NaN,-1.000000,-1.001000,-5.592500
4,early_entry,early_entry__all_open_buys,all selected-wallet open_buy events,fixed,no_fill,4,2026-02-14 09:47:59+00:00,2026-02-14 00:00:00+00:00,NaT,NaN,0xbc70bac4dbd7b8f54634e7aec0507301de7d8f06bb92652c2c992e4d59d99af8,NaN,0x5bbefc673462f1955e31b4a2347450724946c65d,0x39c4330d3495c5733fc7ecdd63ec74cdc0ceb1489dce5e400028bb4bca6f7383,Gen.G,0x39c4330d3495c5733fc7ecdd63ec74cdc0ceb1489dce5e400028bb4bca6f7383|Gen.G,0.010000,1420.630000,NaN,100.000000,0.0,0.0,NaN,0.3125,0.666954,0.3125,1.000000,1.000000,0.881645,11.838583,0,0,0,0,0.00-0.10,False,-0.010000,NaN,NaN,NaN,-1.000000,-1.001000,-14.206300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2348,smooth_pnl,smooth_pnl__score_threshold_0.80,signal_score >= 0.80,kelly_simple,no_fill,559,2026-03-10 14:30:41+00:00,2026-03-10 00:00:00+00:00,NaT,NaN,0xa2cbef5d54cd1dbbfcac98063cd41065a8bf1680dd41c2

In [231]:
all_trigger_ledger.columns

Index(['cohort', 'strategy', 'trigger_rule', 'sizing_mode', 'fill_status',
       'trigger_id', 'dt', 'trade_date', 'tape_dt', 'fill_delay_minutes',
       'trigger_tx_hash', 'fill_tx_hash', 'wallet', 'condition_id', 'outcome',
       'market_key', 'price', 'quantity', 'exec_price', 'stake_usdc',
       'filled_usdc', 'fill_ratio', 'kelly_fraction', 'wallet_quality',
       'signal_score', 'wallet_component', 'conviction_component',
       'price_component', 'consensus_component', 'conviction_ratio',
       'prior_same_any', 'prior_opp_any', 'prior_same_24h', 'prior_opp_24h',
       'price_bucket', 'token_winner', 'prob_edge', 'gross_roi', 'net_roi',
       'net_pnl_usdc', 'trigger_gross_roi', 'trigger_net_roi',
       'trigger_net_pnl_usdc'],
      dtype='object')

In [232]:
# df = dataset.to_table().to_pandas()

In [233]:

# df[df['tx_hash'] == '0xebb91513e879e73d6060eb9f2973e5150bdc5869fa150e88e405d2e4d235b1aa']

In [234]:
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', 1000)
(all_trigger_ledger[(all_trigger_ledger['fill_status'] == 'filled')
                    & (all_trigger_ledger['price'] < 0.9)
                    #& (all_trigger_ledger['exec_price'] - all_trigger_ledger['price'] > 0.1)
                    ]
 [['cohort', 'fill_status', 'dt', 'tape_dt', 'price', 'exec_price', 'wallet_quality', 'token_winner', 'net_pnl_usdc', 'trigger_net_pnl_usdc', 'condition_id', 'trigger_tx_hash', 'fill_tx_hash']]
 .head(10)
)

,cohort,fill_status,dt,tape_dt,price,exec_price,wallet_quality,token_winner,net_pnl_usdc,trigger_net_pnl_usdc,condition_id,trigger_tx_hash,fill_tx_hash
4,quality_core,filled,2026-02-11 01:37:01+00:00,2026-02-11 01:46:55+00:00,0.24,0.211050,0.92,False,-750.750000,NaN,0x5365f6a854d4d74b2c60b1fea5ad662c152965412eb8b9179b83081fda54e072,0x27df9b5abd6d236d117cd623723fba7d66251776403bba05aa496b551de1a0ee,0xced9f287df4d122693de020a886a4d835b2e4dfe5cdf3b562b5ab41331078ee7
535,quality_core,filled,2026-02-11 00:44:19+00:00,2026-02-11 00:45:09+00:00,0.41,0.221100,0.54,False,-100.100000,NaN,0x633b744818a4fd6c58c9daeff8f27e77a2f28d7f11651d48ba1d30da9a14752d,0x711a3820946bf3a595489c84c2d6f06be5b91711d9209fd4bdf0c8c5a41a70a3,0x910f928e783b8efa0ec040827370cb6ead4eb0856a6739b3ebc3cff5932dae64
540,quality_core,filled,2026-02-11 01:37:01+00:00,2026-02-11 01:46:49+00:00,0.24,0.211050,0.92,False,-100.100000,NaN,0x5365f6a854d4d74b2c60b1fea5ad662c152965412eb8b9179b83081fda54e072,0x27df9b5abd6d236d117cd623723fba7d66251776403bba05aa496b551de1a0ee,0x9bf09c85db635d488173f7f73d308c912729677d741ae7c5b1ea35ef86a795b6
543,quality_core,filled,2026-02-11 01:49:15+00:00,2026-02-11 01:50:21+00:00,0.09,0.070350,0.92,False,-100.100000,NaN,0x6433b767e927fdc2ec2e60c70185cd0371d9e793da96ef8489091182c6178ed2,0x6e6e75b4d59400761f9202ab19c9e8de1d27a77409b11ab264bce5c7581d7a13,0x6debefe6a34c712c73e034b850233c600fbdb416a2940c6e0990d96f7ef31db3
553,quality_core,filled,2026-02-12 00:19:37+00:00,2026-02-12 00:22:27+00:00,0.36,0.254232,0.54,False,-100.100000,NaN,0xf83a9761389a0d229324eb425a31b9ed1df9b304d13074e7ffb865c2c90ede7c,0xaca76e7b9716b5f888b65a70cfc99253945f2fc500aa2610cd235c12b2f1c71d,0xb49845d69cc116b3c85cd42f7de65f5e1b6540a2e5c003a74aa2ead9da1f0c4a
594,quality_core,filled,2026-02-14 01:07:27+00:00,2026-02-14 01:07:31+00:00,0.35,0.261300,0.72,True,282.601875,NaN,0xf3a952cd5d10b88704a35b852203097d1f6225d4bbff69d357e4bf267c909fda,0x3114acb915d54758d019f183f013eedea46995d51f914cd55874ac28ed5091bc,0x78f84098854cefa9088bbd43f9e4a5b67841c4e9a87b5ba60e0228b6086cb3a9
612,quality_core,filled,2026-02-15 01:23:19+00:00,2026-02-15 01:27:39+00:00,0.40,0.377324,0.46,False,-100.100000,NaN,0x122d863d3e83b5159165d184964ff0204497e7ac462dbe4c602d5ccc40ca8e63,0x0e25811a7c4c53d81db71c345f8b5f597cfe5a6065cd4ed4cf39f7d08154dde2,0x151216b73ad7eb189e4c07f419f58f29494e7cc1c8fd02e4570690c16adfc366
632,quality_core,filled,2026-02-16 03:42:41+00:00,2026-02-16 03:43:21+00:00,0.11,0.014280,0.82,False,-100.100000,NaN,0xf2075cae5885590c67600c29c593e1bfeb1fea12e2943f154bede4ceb94f40c7,0x4fa78bc7fb3f2bd60401af7da52eec3815449360725f295a19b8875a7c5ec0dd,0x2606ea81563a9af0d8d10cf0ba64e7f07c1bc238d081cf23791f5f68b2a040c2
633,quality_core,filled,2026-02-16 04:10:23+00:00,2026-02-16 04:12:15+00:00,0.32,0.280407,0.54,False,-100.100000,NaN,0xd6aa9e138ca022a6cd3cb4e93fe1719de5331d3109f61a6a9b1304e6afb6696d,0x1f56834cfcba6e9ca1468645ccc77bcbac79b2a5fe98cd22a37c8efd47e0f730,0x81194433dddf5aad7cd23e70004f8f5e70de5a013e305490442cf2902187a27f
634,quality_core,filled,2026-02-16 05:04:33+00:00,2026-02-16 05:04:37+00:00,0.20,0.114924,0.82,False,-100.100000,NaN,0x382aed04fd32aa35d057e5a08e270f0176d2e5b2a178fd843c2b02bdb4903242,0xb2b9218f104fd216335b65396ec88f8ebb6dbe2b995a602578153b75d0a51a40,0x56dba4082b52b23f2d2c399c9c5fde95eb64766ccd862a2575bcef8d2e32b719


## 7. Robustness checks and benchmark interpretation

In [235]:
monthly_by_strategy = []
for name, run in strategy_runs.items():
    trades = run['trades']
    if trades.empty:
        continue
    part = trades.assign(month=trades['trade_date'].dt.to_period('M').astype(str)).groupby('month').agg(
        trades=('market_key', 'size'),
        net_pnl_usdc=('net_pnl_usdc', 'sum'),
        avg_net_roi=('net_roi', 'mean'),
        win_rate=('token_winner', 'mean'),
    ).reset_index()
    part['strategy'] = name
    monthly_by_strategy.append(part)

monthly_by_strategy = pd.concat(monthly_by_strategy, ignore_index=True) if monthly_by_strategy else pd.DataFrame()
monthly_by_strategy.sort_values(['month', 'net_pnl_usdc'], ascending=[True, False])


/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_4834/529857251.py:6: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.

/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_4834/529857251.py:6: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.

/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_4834/529857251.py:6: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.

/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_4834/529857251.py:6: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.

/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_4834/529857251.py:6: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.

/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_4834/529857251.py:6: UserWarning:

Converting to PeriodArray/Index repr

,month,trades,net_pnl_usdc,avg_net_roi,win_rate,strategy
1,2026-02,20,1807.716522,0.903858,0.300000,quality_core__all_open_buys
4,2026-02,9,-348.310674,-0.387012,0.444444,early_entry__all_open_buys
0,2026-02,1,-750.750000,-1.001000,0.000000,quality_core__score_threshold_0.80
8,2026-02,74,-889.912374,-0.120258,0.472973,smooth_pnl__all_open_buys
6,2026-02,78,-8508.277915,-0.103631,0.397436,smooth_pnl__score_threshold_0.80
5,2026-03,4,1174.759344,2.936898,0.750000,early_entry__all_open_buys
2,2026-03,10,-460.964763,-0.460965,0.300000,quality_core__all_open_buys
9,2026-03,46,-603.480346,-0.131191,0.434783,smooth_pnl__all_open_buys
3,2026-03,1,-750.750000,-1.001000,0.000000,early_entry__score_threshold_0.80
7,2026-03,27,-1215.555515,-0.096121,0.407407,smooth_pnl__score_threshold_0.80


In [236]:
test_score_bins = test_scored.assign(score_decile=pd.qcut(test_scored['signal_score'].rank(method='first'), 10, labels=False))
test_score_bins.groupby('score_decile').agg(
    signals=('wallet', 'size'),
    avg_prob_edge=('prob_edge', 'mean'),
    avg_copy_roi_capped=('copy_roi_capped', 'mean'),
    win_rate=('token_winner', 'mean'),
)


,signals,avg_prob_edge,avg_copy_roi_capped,win_rate
score_decile,,,,
0,294,-0.243199,-0.057770,0.421769
1,294,0.089216,0.354237,0.469388
2,293,0.164374,0.763281,0.457338
3,294,0.184610,1.125074,0.421769
4,293,0.163224,0.681423,0.474403
5,294,0.055923,-0.066666,0.125850
6,293,0.362223,3.008455,0.464164
7,294,0.364305,3.295109,0.442177
8,293,0.292339,2.124097,0.392491


In [237]:
rng = np.random.default_rng(42)
eligible_placebo = full_train_metrics[
    (full_train_metrics['open_buy_trades'] >= 20)
    & (full_train_metrics['volume'] >= 500.0)
    & (full_train_metrics['distinct_markets'] >= 8)
    & (full_train_metrics['recent_open_buy_trades'] >= 3)
].copy()

if len(eligible_placebo) >= len(selected_wallets):
    placebo_wallets = eligible_placebo.sample(n=len(selected_wallets), random_state=42).copy().reset_index(drop=True)
    placebo_wallets['wallet_quality'] = rng.random(len(placebo_wallets))
    placebo_profiles = build_wallet_profiles(dataset, placebo_wallets, period='full_train')
    _, placebo_test_open_buys = build_signal_events(dataset, placebo_profiles, period='test')
    placebo_scored = apply_signal_score(placebo_test_open_buys, price_table, consensus_table)
    placebo_trades_bt, placebo_daily_bt, placebo_unfilled_bt, placebo_theoretical_daily_bt = backtest_strategy(
        placebo_scored,
        mask=placebo_scored['signal_score'] >= SIGNAL_THRESHOLD,
        tape_groups=tape_groups,
        strategy_name='placebo_random_wallets',
        trigger_rule=f'placebo signal_score >= {SIGNAL_THRESHOLD:.2f}',
        base_stake_usdc=100.0,
        dynamic_sizing=True,
        latency_seconds=0,
        fill_horizon_seconds=600,
        slippage_bps=50.0,
        fee_bps=10.0,
        max_signals_per_day=20,
        dedupe_by_market=True,
        starting_capital=10000.0,
        cohort_name='placebo',
    )
    placebo_compare = pd.DataFrame([
        summarize_backtest(test_trades_bt, test_daily_bt).rename(SCORE_STRATEGY_NAME),
        summarize_backtest(placebo_trades_bt, placebo_daily_bt).rename('placebo_random_wallets'),
    ])
    placebo_compare['unfilled_triggers'] = [len(score_unfilled_trigger_ledger), len(placebo_unfilled_bt)]
    placebo_compare
else:
    pd.DataFrame({'note': ['Not enough eligible wallets for placebo sample']})


In [238]:
def with_zero_anchor(daily):
    if daily.empty:
        return daily
    first_date = daily['trade_date'].min()
    anchor = pd.DataFrame({'trade_date': [first_date - pd.Timedelta(days=1)], 'net_pnl_usdc': [0.0], 'cum_net_pnl_usdc': [0.0]})
    return pd.concat([anchor, daily], ignore_index=True).sort_values('trade_date').reset_index(drop=True)

fig = make_subplots(rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.12, subplot_titles=('Benchmark comparison cumulative PnL (train ref + test)', 'Train-B score calibration'))

for name, run in strategy_runs_train_ref.items():
    daily = run['daily']
    if not daily.empty:
        plot_daily = with_zero_anchor(daily)
        fig.add_trace(
            go.Scatter(
                x=plot_daily['trade_date'],
                y=plot_daily['cum_net_pnl_usdc'],
                mode='lines',
                line={'dash': 'dot'},
                name=f'{name} [train_ref filled]',
                opacity=0.65,
            ),
            row=1,
            col=1,
        )
    theoretical_daily = run.get('theoretical_daily', pd.DataFrame())
    if not theoretical_daily.empty:
        plot_theoretical = with_zero_anchor(theoretical_daily)
        fig.add_trace(
            go.Scatter(
                x=plot_theoretical['trade_date'],
                y=plot_theoretical['cum_net_pnl_usdc'],
                mode='lines',
                line={'dash': 'dashdot'},
                name=f'{name} [train_ref trigger_theoretical]',
                opacity=0.45,
            ),
            row=1,
            col=1,
        )

for name, run in strategy_runs.items():
    daily = run['daily']
    if not daily.empty:
        plot_daily = with_zero_anchor(daily)
        fig.add_trace(
            go.Scatter(
                x=plot_daily['trade_date'],
                y=plot_daily['cum_net_pnl_usdc'],
                mode='lines',
                name=f'{name} [test filled]',
            ),
            row=1,
            col=1,
        )
    theoretical_daily = run.get('theoretical_daily', pd.DataFrame())
    if not theoretical_daily.empty:
        plot_theoretical = with_zero_anchor(theoretical_daily)
        fig.add_trace(
            go.Scatter(
                x=plot_theoretical['trade_date'],
                y=plot_theoretical['cum_net_pnl_usdc'],
                mode='lines',
                line={'dash': 'dashdot'},
                name=f'{name} [test trigger_theoretical]',
                opacity=0.55,
            ),
            row=1,
            col=1,
        )

split_dt = pd.Timestamp(END_DATE_TRAIN) + pd.Timedelta(days=0.5)
fig.add_vline(x=split_dt, line_dash='dash', line_color='black', row=1, col=1)
fig.add_annotation(x=split_dt, y=1.02, yref='paper', text='train/test split', showarrow=False)

cal_plot = score_deciles.groupby('score_decile').agg(avg_copy_roi_capped=('copy_roi_capped', 'mean')).reset_index()
fig.add_trace(go.Bar(x=cal_plot['score_decile'], y=cal_plot['avg_copy_roi_capped'], name='train_b avg copy roi capped'), row=2, col=1)
fig.update_layout(template='plotly_white', height=900, title='Wallet signal v2 benchmark diagnostics')
fig.show(renderer='browser')


## Notes

Intentional choices in v2:
- skill is based on fresh entries only, so ranking matches the event we actually trade
- hit rate stays only as a benchmark
- `avg_copy_roi_capped` is capped to stop ultra-cheap tails from dominating ranking by luck
- the score is simple and auditable; if it works, the next upgrade is a train-only model using these same features
